In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('raw_aegis_payroll_2023.csv')

# Check the exact number of rows and columns
print(f"Dataset Shape: {df.shape}\n")
display(df.head())

C:\Users\Bea Gamilong\AppData\Local\Temp\ipykernel_30988\2305296473.py:4: DtypeWarning: Columns (0: Regular Hours, 1: OT Hours) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('raw_aegis_payroll_2023.csv')


Dataset Shape: (552967, 17)



,Fiscal Year,Payroll Number,Agency Name,Last Name,First Name,Mid Init,Agency Start Date,Work Location Borough,Title Description,Leave Status as of June 30,Base Salary,Pay Basis,Regular Hours,Regular Gross Paid,OT Hours,Total OT Paid,Total Other Pay
0,2023,67.0,ADMIN FOR CHILDREN'S SVCS,ZAFFARANO,CHRISTINE,NaN,04/11/2016,MANHATTAN,AGENCY ATTORNEY,CEASED,"$91,563.00",per Annum,35,"$1,755.85",0,$0.00,$95.05
1,2023,67.0,ADMIN FOR CHILDREN'S SVCS,WRIGHT,TREMAINE,O,08/29/2016,MANHATTAN,COMMUNITY COORDINATOR,ACTIVE,"$78,450.00",per Annum,"1,820","$78,361.19",0,$0.00,"$3,000.00"
2,2023,67.0,ADMIN FOR CHILDREN'S SVCS,XAVIER,CALLYANN,M,05/22/2017,BROOKLYN,CHILD PROTECTIVE SPECIALIST,ACTIVE,"$65,921.00",per Annum,"1,820","$65,998.30",285.75,"$13,732.03","$12,901.84"
3,2023,67.0,ADMIN FOR CHILDREN'S SVCS,WILSON,DESIREE,NaN,02/06/2017,BROOKLYN,CHILD PROTECTIVE SPECIALIST,ACTIVE,"$65,921.00",per Annum,"1,820","$65,791.47",41,"$1,595.59","$12,672.67"
4,2023,67.0,ADMIN FOR CHILDREN'S SVCS,WILSON,GEOFFREY,M,06/15/2009,BROOKLYN,CHILD PROTECTIVE SPECIALIST,ACTIVE,"$65,921.00",per Annum,"1,820","$65,837.61",0,$9.89,"$14,532.71"


In [4]:
# Identify missing values
missing_values = df.isnull().sum()
print("Null Values per column): ")
print(missing_values[missing_values > 0])

Null Values per column): 
Payroll Number                3
Last Name                   362
First Name                  374
Mid Init                 230993
Work Location Borough         1
Title Description             3
dtype: int64


In [5]:
# Check for fragmented department naming conventions
# Printing top 50 to spot duplicates (e.g., "POLICE DEPT" vs "DEPT OF POLICE")
agency_counts = df['Agency Name'].value_counts()
display(agency_counts.head(50))

Agency Name
DEPT OF ED PEDAGOGICAL            106882
DEPT OF ED PER SESSION TEACHER     82883
POLICE DEPARTMENT                  58623
DEPT OF ED PARA PROFESSIONALS      39580
BOARD OF ELECTION POLL WORKERS     29920
DEPT OF ED HRLY SUPPORT STAFF      26901
FIRE DEPARTMENT                    19075
DEPARTMENT OF EDUCATION ADMIN      17551
DEPT OF PARKS & RECREATION         16739
DEPT OF ED PER DIEM TEACHERS       15020
NYC HOUSING AUTHORITY              14672
HRA/DEPT OF SOCIAL SERVICES        13908
DEPARTMENT OF SANITATION           11430
DEPARTMENT OF CORRECTION           10362
ADMIN FOR CHILDREN'S SVCS           8861
DEPT OF HEALTH/MENTAL HYGIENE       8225
DEPT OF ENVIRONMENT PROTECTION      6971
DEPARTMENT OF TRANSPORTATION        6843
COMMUNITY COLLEGE (MANHATTAN)       4267
COMMUNITY COLLEGE (LAGUARDIA)       3264
HOUSING PRESERVATION & DVLPMNT      3113
DEPT OF CITYWIDE ADMIN SVCS         2708
COMMUNITY COLLEGE (KINGSBORO)       2650
COMMUNITY COLLEGE (QUEENSBORO)      2325
DIST

In [6]:
# Hunt for middle initials that contain numbers or special characters
invalid_mid_init = df[df['Mid Init'].str.contains(r'[^a-zA-Z]', na=False, regex=True)]

print(f"Found {len(invalid_mid_init)} rows with invalid Middle Initials (Numbers/Special Chars).")

# Display a sample of the broken rows
display(invalid_mid_init[['Agency Name', 'Title Description', 'Mid Init']].head(10))

Found 41 rows with invalid Middle Initials (Numbers/Special Chars).


,Agency Name,Title Description,Mid Init
10779,BOARD OF ELECTION POLL WORKERS,ELECTION WORKER,""""
26039,BOARD OF ELECTION POLL WORKERS,ELECTION WORKER,-
27661,BOARD OF ELECTION POLL WORKERS,ELECTION WORKER,0
34336,BOARD OF ELECTION POLL WORKERS,ELECTION WORKER,&
44517,COMMUNITY COLLEGE (BRONX),CUNY OFFICE ASSISTANT,1
45182,COMMUNITY COLLEGE (BRONX),LECTURER,2
54923,COMMUNITY COLLEGE (MANHATTAN),ADJUNCT LECTURER,1
113480,DEPARTMENT OF TRANSPORTATION,CLERICAL ASSOCIATE,-
117036,DEPT OF CITYWIDE ADMIN SVCS,ARCHITECT,-
121989,DEPT OF ED HRLY SUPPORT STAFF,COOP EDUCATION TRAINEE COMPUTER TECH AIDE/JR,0


In [8]:
# Hunt for dates with inconsistent data types (there might be strings)
# Count the exact Python data types inside the Agency Start Date column
print("Data Type Distribution in 'Agency Start Date':")
type_counts = df['Agency Start Date'].apply(type).value_counts()
print(type_counts)

# Side-by-side comparison of the value and its type
print("\nSide-by-side Sample of Values and their Hidden Types:")
sample_types = df[['Agency Start Date']].dropna().sample(10).copy()

sample_types['Underlying Type'] = sample_types['Agency Start Date'].apply(lambda x: type(x).__name__)

display(sample_types)

Data Type Distribution in 'Agency Start Date':
Agency Start Date
<class 'str'>    552967
Name: count, dtype: int64

Side-by-side Sample of Values and their Hidden Types:


,Agency Start Date,Underlying Type
45113,05/12/2012,str
505462,09/22/2008,str
468110,04/17/2023,str
532539,11/16/2022,str
442772,09/17/2018,str
431277,07/01/2004,str
283409,01/22/2008,str
68360,08/12/2004,str
26659,01/01/2021,str
160011,12/02/2002,str


In [10]:
# 1. Define the financial columns
financial_cols = ['Regular Hours', 'Regular Gross Paid', 'OT Hours', 'Total OT Paid']

# 2. Force them into numeric data types so we can do math on them
for col in financial_cols:
    # Remove any potential commas first, just in case
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '', regex=True)
    
    # Convert to numeric, turning weird text into NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Display the min and max values to catch negative numbers and massive outliers
display(df[financial_cols].describe().loc[['min', 'max']])

# 4. Isolate the top 10 most extreme burnout cases
burnout_outliers = df.sort_values(by='OT Hours', ascending=False)[['Agency Name', 'Title Description', 'OT Hours', 'Total OT Paid']]
display(burnout_outliers.head(10))

,Regular Hours,Regular Gross Paid,OT Hours,Total OT Paid
min,-1529.27,NaN,-96.75,NaN
max,6666.00,NaN,3172.50,NaN


,Agency Name,Title Description,OT Hours,Total OT Paid
465246,HRA/DEPT OF SOCIAL SERVICES,ELIGIBILITY SPECIALIST,3172.50,NaN
397858,DEPT OF HEALTH/MENTAL HYGIENE,INSTITUTIONAL AIDE,3047.25,NaN
403104,DEPT OF HEALTH/MENTAL HYGIENE,INSTITUTIONAL AIDE,3044.50,NaN
401351,DEPT OF HEALTH/MENTAL HYGIENE,SOCIAL WORKER,2816.75,NaN
463687,HRA/DEPT OF SOCIAL SERVICES,SUPERVISOR I,2736.50,NaN
405130,DEPT. OF HOMELESS SERVICES,CITY LABORER,2684.00,NaN
461944,HRA/DEPT OF SOCIAL SERVICES,BOOKKEEPER,2640.50,NaN
403838,DEPT OF HEALTH/MENTAL HYGIENE,INSTITUTIONAL AIDE,2553.75,NaN
457998,HRA/DEPT OF SOCIAL SERVICES,SUPERVISOR I,2543.00,NaN
73561,DEPARTMENT OF CORRECTION,MARINE OILER,2512.00,NaN
